# Chapter 8 — Data Modification SQL Notebook | Northwinds2024Student Database

**Course:** CSCI 331

**Name:** Simran Singh

**Assignment:** <span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);"> HW8 — Individual Propositions &amp; Querie</span>

**Database:** <span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">Northwinds2024Student</span>

**Tool:** Azure Data Studio

* * *

### Overview

This notebook contains 10 propositions and their matching T-SQL queries covering Chapter 8 topics: INSERT, SELECT INTO, DELETE with OUTPUT, UPDATE with OUTPUT, and JOIN-based modifications. All queries are tested against the Northwinds2024Student named database

---
## Proposition 1 — INSERT: Add a New Customer Manually

**Problem Statement:**  
Sometimes a new customer gets added before they even place their first order. I want to insert a single customer record directly into Sales.Customer to make sure they're in the system and ready to go when their first order comes in.

**Why this query is special:**  
This is the most basic form of data modification — a direct single-row INSERT. It's important to understand this before moving on to bulk inserts because it shows how each column maps to a value and what constraints the table enforces.

In [2]:
-- Proposition 1: INSERT a single new customer
USE Northwinds2024Student;

INSERT INTO Sales.Customer (CustomerId, CustomerCompanyName, CustomerCountry, CustomerRegion, CustomerCity)
VALUES (100, 'Coho Winery', 'USA', 'WA', 'Redmond');

-- Verify the insert
SELECT * FROM Sales.Customer WHERE CustomerId = 100;

(0 rows affected)

Total execution time: 00:00:00.056

CustomerId,CustomerCompanyName,CustomerContactName,CustomerContactTitle,CustomerAddress,CustomerCity,CustomerRegion,CustomerPostalCode,CustomerCountry,CustomerPhoneNumber,CustomerFaxNumber


: Msg 544, Level 16, State 1, Line 4
Cannot insert explicit value for identity column in table 'Customer' when IDENTITY_INSERT is set to OFF.

* * *

## Proposition 2 — INSERT with SELECT: Bring in Customers Who Have Orders

**Problem Statement:**  
I only want customers in my working table who have actually placed an order — no point having records for people who never bought anything. So I'm pulling from Sales.Customer but filtering to only the ones that show up in Sales.Order.

**Why this query is special:**  
This combines INSERT with a subquery using EXISTS. It's more efficient than a JOIN for this use case because EXISTS stops looking as soon as it finds the first matching row, which matters on large datasets.

In [4]:
-- Proposition 2: INSERT customers who have placed at least one order
USE Northwinds2024Student;

-- Setup: create working table first
DROP TABLE IF EXISTS dbo.Customers;
CREATE TABLE dbo.Customers
(
  CustomerId      INT           NOT NULL PRIMARY KEY,
  companyname NVARCHAR(40)  NOT NULL,
  country     NVARCHAR(15)  NOT NULL,
  region      NVARCHAR(15)  NULL,
  city        NVARCHAR(15)  NOT NULL
);

INSERT INTO dbo.Customers (CustomerId, companyname, country, region, city)
SELECT DISTINCT C.CustomerId, C.CustomerCompanyName, C.CustomerCountry, C.CustomerRegion, C.CustomerRegion
FROM Sales.Customer AS C
WHERE EXISTS (
    SELECT 1 FROM Sales.[Order] AS O
    WHERE O.CustomerId = C.CustomerId
);

-- Verify
SELECT COUNT(*) AS CustomersWithOrders FROM dbo.Customers;

The statement has been terminated.

(1 row affected)

Total execution time: 00:00:00.062

CustomersWithOrders
0


: Msg 515, Level 16, State 2, Line 15
Cannot insert the value NULL into column 'city', table 'Northwinds2024Student.dbo.Customers'; column does not allow nulls. INSERT fails.

---
## Proposition 3 — SELECT INTO: Build a Focused Orders Table by Date Range

**Problem Statement:**  
The full Sales.Order table has way too much data to work with efficiently. I want to spin up a new table that only holds orders from 2020 through 2022 so I can run analysis without touching the original data.

**Why this query is special:**  
SELECT INTO creates and populates a table in one step — no need to define the schema first. This is really useful for building working or temp tables quickly during analysis.

In [5]:
-- Proposition 3: SELECT INTO to create a filtered Orders working table
USE Northwinds2024Student;

DROP TABLE IF EXISTS dbo.Orders;

SELECT *
INTO dbo.Orders
FROM Sales.[Order]
WHERE orderdate >= '20200101'
  AND orderdate < '20230101';

-- Verify the new table
SELECT COUNT(*) AS OrdersCopied FROM dbo.Orders;
SELECT MIN(orderdate) AS EarliestOrder, MAX(orderdate) AS LatestOrder FROM dbo.Orders;

(830 rows affected)

(1 row affected)

(1 row affected)

Total execution time: 00:00:00.077

OrdersCopied
830


EarliestOrder,LatestOrder
2020-07-04,2022-05-06


---
## Proposition 4 — DELETE with OUTPUT: Clean Out Orders Before August 2020

**Problem Statement:**  
Old orders from before August 2020 need to come out of my working table. But before I delete anything I want to capture what's being removed using OUTPUT — that way if someone asks what got deleted I actually have an answer.

**Why this query is special:**  
The OUTPUT clause is what makes this more than just a basic DELETE. Being able to see exactly what rows were removed — without a separate SELECT beforehand — is a huge deal for auditing and data integrity.

In [6]:
-- Proposition 4: DELETE with OUTPUT to capture removed orders
USE Northwinds2024Student;

DELETE FROM dbo.Orders
OUTPUT
    deleted.orderid,
    deleted.orderdate
WHERE orderdate < '20200801';

-- Confirm remaining records start from August 2020
SELECT MIN(orderdate) AS EarliestRemaining FROM dbo.Orders;

(22 rows affected)

(1 row affected)

Total execution time: 00:00:00.038

orderid,orderdate
10248,2020-07-04
10249,2020-07-05
10250,2020-07-08
10251,2020-07-08
10252,2020-07-09
10253,2020-07-10
10254,2020-07-11
10255,2020-07-12
10256,2020-07-15
10257,2020-07-16


EarliestRemaining
2020-08-01


---
## Proposition 5 — DELETE with JOIN: Remove OrderDetails Tied to Brazilian Suppliers

**Problem Statement:**  
If a supplier gets removed from the system, the order details linked to their products become irrelevant. I'm joining Sales.OrderDetail back to Production.Supplier to find and remove those records cleanly.

**Why this query is special:**  
You can't delete from Sales.OrderDetail directly based on supplier info — that data lives in a completely different schema. This query chains three tables together in a DELETE, which is a more advanced pattern that requires understanding how FKs connect across schemas.

In [8]:
-- Proposition 5: DELETE with multi-table JOIN across schemas
USE Northwinds2024Student;

-- Preview what will be deleted first
SELECT OD.OrderId, OD.ProductId, S.SupplierCompanyName AS SupplierName, S.SupplierCountry
FROM Sales.OrderDetail AS OD
JOIN Production.Product AS P ON OD.productid = P.productid
JOIN Production.Supplier AS S ON P.SupplierId = S.supplierid
WHERE S.SupplierCountry = 'Brazil';

-- Execute the delete
DELETE OD
FROM Sales.OrderDetail AS OD
JOIN Production.Product AS P ON OD.productid = P.productid
JOIN Production.Supplier AS S ON P.supplierid = S.supplierid
WHERE S.SupplierCountry = 'Brazil';

(51 rows affected)

(51 rows affected)

Total execution time: 00:00:00.082

OrderId,ProductId,SupplierName,SupplierCountry
10254,24,Supplier UNAHG,Brazil
10263,24,Supplier UNAHG,Brazil
10275,24,Supplier UNAHG,Brazil
10280,24,Supplier UNAHG,Brazil
10281,24,Supplier UNAHG,Brazil
10293,24,Supplier UNAHG,Brazil
10352,24,Supplier UNAHG,Brazil
10355,24,Supplier UNAHG,Brazil
10358,24,Supplier UNAHG,Brazil
10386,24,Supplier UNAHG,Brazil


---
## Proposition 6 — UPDATE with OUTPUT: Fix NULL Regions in Customer Table

**Problem Statement:**  
I noticed a bunch of NULL values sitting in the region column of Sales.Customer. That breaks reports and filters, so I'm replacing them all with `<None>`. I'm also using OUTPUT so I can see exactly which customers got updated and what changed.

**Why this query is special:**  
The OUTPUT clause on an UPDATE lets you see both the old value (deleted.region) and the new value (inserted.region) side by side. This is really useful for validation — you can confirm the change happened correctly without running a follow-up SELECT.

In [16]:
-- Proposition 6: UPDATE with OUTPUT showing before and after values
USE Northwinds2024Student;

UPDATE Sales.Customer
SET CustomerRegion = '<None>'
OUTPUT
    deleted.CustomerId,
    deleted.CustomerRegion AS oldregion,
    inserted.CustomerRegion AS newregion
WHERE CustomerRegion IS NULL;

(60 rows affected)

Total execution time: 00:00:00.041

CustomerId,oldregion,newregion
1,NULL,<None>
2,NULL,<None>
3,NULL,<None>
4,NULL,<None>
5,NULL,<None>
6,NULL,<None>
7,NULL,<None>
8,NULL,<None>
9,NULL,<None>
11,NULL,<None>


---
## Proposition 7 — UPDATE with JOIN: Sync Shipping Info from Customer Records

**Problem Statement:**  
Some orders have shipping details that don't match what's actually on the customer record — probably from old data entry. I'm fixing that by joining Sales.Order to Sales.Customer and overwriting the ship fields with the correct current values.

**Why this query is special:**  
This is a cross-table UPDATE using FROM and JOIN — one of the most practical patterns in data modification. It's the kind of thing you'd actually use in a real data cleanup job, not just in a textbook exercise.

In [18]:
-- Proposition 7: UPDATE with JOIN to sync shipping fields from customer data
USE Northwinds2024Student;

UPDATE O
SET O.shipToCountry = C.CustomerCountry,
    O.ShipToRegion  = C.CustomerRegion,
    O.ShipToCity    = C.CustomerCity
FROM Sales.[Order] AS O
JOIN Sales.Customer AS C
  ON O.CustomerId = C.CustomerId
WHERE C.CustomerCountry = 'UK';

(56 rows affected)

Total execution time: 00:00:00.009

---
## Proposition 8 — DELETE: Respecting Foreign Key Constraints (Child Before Parent)

**Problem Statement:**  
When I tried to delete from Sales.Order while Sales.OrderDetail still had rows pointing to it, I got a foreign key violation error. You have to clear the child table first. This proposition demonstrates the correct sequence so the operation actually works.

**Why this query is special:**  
This one isn't flashy but it's one of the most common mistakes people make with relational databases. Understanding referential integrity and delete order is critical before doing any large-scale data modification work.

In [12]:
-- Proposition 8: Correct order of deletion to respect FK constraints
USE Northwinds2024Student;

-- Step 1: Delete from child table first (FK references Orders)
DELETE FROM Sales.OrderDetail
WHERE orderid IN (
    SELECT orderid FROM dbo.Orders
    WHERE YEAR(orderdate) < 2020
);

-- Step 2: Now safe to delete from parent table
DELETE FROM dbo.Orders
WHERE YEAR(orderdate) < 2020;

-- Verify cleanup
SELECT COUNT(*) AS RemainingOrders FROM dbo.Orders;

(0 rows affected)

(0 rows affected)

(1 row affected)

Total execution time: 00:00:00.043

RemainingOrders
808


---
## Proposition 9 — INSERT with Filter: Copy Only High-Freight Orders

**Problem Statement:**  
For a shipping cost analysis I only care about orders where freight was over $100. Instead of filtering every time I query, I'm inserting just those rows into a working table upfront to make future queries faster and cleaner.

**Why this query is special:**  
This is a good example of using INSERT with a filtered SELECT to build a purpose-built reporting table. It's more efficient than repeatedly filtering the full Orders table and makes the downstream analysis much simpler.

In [13]:
-- Proposition 9: INSERT filtered rows into a high-freight working table
USE Northwinds2024Student;

DROP TABLE IF EXISTS dbo.HighFreightOrders;

SELECT *
INTO dbo.HighFreightOrders
FROM Sales.[Order]
WHERE freight > 100;

-- Summary of what was captured
SELECT
    COUNT(*)        AS TotalOrders,
    MIN(freight)    AS MinFreight,
    MAX(freight)    AS MaxFreight,
    AVG(freight)    AS AvgFreight
FROM dbo.HighFreightOrders;

(187 rows affected)

(1 row affected)

Total execution time: 00:00:00.041

TotalOrders,MinFreight,MaxFreight,AvgFreight
187,100.22,1007.64,233.0429


---
## Proposition 10 — UPDATE with CASE: Bucket Customers into Sales Regions

**Problem Statement:**  
After cleaning up the NULL regions I wanted to go a step further and group customers into broader geographic zones like Americas or Europe. This makes it way easier to slice sales reports by territory without having to write long WHERE clauses every time.

**Why this query is special:**  
Using a CASE expression inside an UPDATE is one of those techniques that feels advanced but is actually very practical. It lets you apply conditional logic across all rows in a single pass — no loops, no cursors, just clean set-based SQL.

In [14]:
-- Proposition 10: UPDATE with CASE to assign geographic sales regions
USE Northwinds2024Student;

UPDATE dbo.Customers
SET region = CASE
    WHEN country IN ('USA', 'Canada', 'Mexico')                          THEN 'Americas'
    WHEN country IN ('UK', 'Germany', 'France', 'Spain', 'Sweden',
                     'Italy', 'Portugal', 'Denmark', 'Finland',
                     'Norway', 'Belgium', 'Switzerland', 'Austria',
                     'Ireland', 'Poland')                                THEN 'Europe'
    WHEN country IN ('Brazil', 'Argentina', 'Venezuela')                 THEN 'South America'
    WHEN region = '<None>'                                               THEN 'Other'
    ELSE region
END;

-- See the distribution of customers across regions
SELECT region, COUNT(*) AS CustomerCount
FROM dbo.Customers
GROUP BY region
ORDER BY CustomerCount DESC;

-- Cleanup (run when done)
-- DROP TABLE IF EXISTS dbo.Customers, dbo.Orders, dbo.HighFreightOrders;

(0 rows affected)

(0 rows affected)

Total execution time: 00:00:00.020

region,CustomerCount
